# LOSO on Google Colab (P21): leave-one-subject-out SSVEP classification

This notebook trains on every subject but one and tests on the held-out subject, for each subject in turn.
It also runs the within-subject reference and prints a table that compares the two.
Run the cells **from top to bottom, one at a time** (click a cell, press Shift+Enter).
You do not need to understand the code to run it; you only edit the **settings cell** (step 3).

**What you need before you start**
1. A Google account with Google Drive and, for the networks, a Colab GPU (the credits you were given).
2. The recordings uploaded to your Drive (this cannot be automated): one folder per subject, for example
   `MyDrive/ssvep/raw/S01/OO.ebr`, `MyDrive/ssvep/raw/S02/OO.ebr`, and so on. Upload them from the Drive web page
   (New > Folder upload). Do not rename the files.
3. Choose the runtime: menu **Runtime > Change runtime type > T4 GPU** (or any GPU) for the whole notebook.
   The classical part (steps 7 to 9) does not need a GPU, but the networks (step 10) do.

**Expected runtimes for 40 subjects** (measured on an RTX 3050-class GPU; a Colab T4 is of a similar order, expect
the same figures within a factor of about two):

| Step | Time |
|---|---|
| Build windows (first time) | a few minutes up to about an hour for 40 recordings; skipped when already built |
| Within-subject reference | minutes |
| LOSO classical (`fbcca`, `lda`, `lda_subjnorm`) | about 30 to 35 min |
| LOSO network `ccnn` | about 1.5 h |
| LOSO network `eegnet` | about 4.9 h |
| LOSO network `compact_eegnet` | about 12.5 h |

A free or Pro session is cut after some hours, or when the browser tab is idle. **Nothing is lost**: every held-out
subject (a "fold") is saved to Drive when it finishes. After a disconnect, open the notebook again, run steps 1 to 6
once more, then rerun the cell you were in; finished folds print `[skip]` and only the missing ones are computed.
Run one network per cell, and do not start the next until the previous one is finished.

## 1. Check the GPU
You should see a line with `Tesla T4` (or another GPU). If it says "command not found" or "failed", choose
**Runtime > Change runtime type**, pick a GPU, and run this cell again (the runtime restarts; that is fine).

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 2. Connect your Google Drive
A pop-up asks for permission to access your Drive. Accept it. Everything that must survive a disconnect
(built windows, results, trained models) is stored on Drive. The Colab disk itself is wiped when the session ends.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Settings (the only cell you edit)
- `RAW_DIR`: the Drive folder that holds the `S01`, `S02`, ... folders with the recordings.
- `DRIVE_DIR`: a Drive folder where this notebook keeps its outputs (it is created if missing).
- `REPO_URL` and `BRANCH`: the code to run. Leave them unless you are told otherwise.
- `LOSO_OUT`: the results folder of this configuration. **Use a new name whenever you change a setting**
  (window length, stride, channels, seed), for example `reports/loso_w2_colab`. A folder remembers its
  configuration and refuses to mix folds computed with another one.

The cell prints the subject folder names it finds and whether each has an `OO.ebr` file. It never prints
recording content. Fix any `MISSING` line before going on.

In [ ]:
import os
import re

RAW_DIR = "/content/drive/MyDrive/ssvep/raw"
DRIVE_DIR = "/content/drive/MyDrive/ssvep/colab_out"
REPO_URL = "https://github.com/Upland1/mas-ssvep-xai.git"
BRANCH = "main"
LOSO_OUT = "reports/loso_w1_colab"

REPO_DIR = "/content/repo"
WITHIN_OUT = "reports/window_sweep_colab"
MODELS_DIR = "outputs/models/" + os.path.basename(LOSO_OUT)

subjects = sorted((d for d in os.listdir(RAW_DIR) if re.fullmatch(r"S\d+", d)), key=lambda s: int(s[1:]))
print(len(subjects), "subject folders:", " ".join(subjects))
missing = [s for s in subjects if not os.path.isfile(os.path.join(RAW_DIR, s, "OO.ebr"))]
for s in missing:
    print("MISSING OO.ebr in", s)
assert subjects, "no S<number> folders in RAW_DIR: check the path and the upload"
assert not missing, "fix the MISSING subjects above (upload OO.ebr or remove the folder)"
print("settings OK")

## 4. Download the code (fresh copy every session)
The Colab disk is empty at each new session, so the code is cloned again. Re-running this cell deletes the previous
copy of the code (not your results, which live on Drive) and clones it again.

In [ ]:
!rm -rf {REPO_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

## 5. Install the two missing packages
Colab already has PyTorch, NumPy, SciPy, scikit-learn and pandas. Only `pyyaml` and `joblib` are added.
**Never install the whole requirements file here** (the `-r` form of pip): it can replace the GPU build of PyTorch
that Colab ships and break the GPU.

In [ ]:
!pip install -q pyyaml joblib

## 6. Link the data and the outputs to Drive
The recordings folder, the built windows (`data/processed_continuous`), `reports` and `outputs/models` become
shortcuts to folders on Drive, so the code writes straight to Drive and nothing is lost when the session ends.

In [ ]:
import os
import shutil
os.makedirs(DRIVE_DIR, exist_ok=True)
links = {
    "data/" + "raw": RAW_DIR,
    "data/processed_continuous": DRIVE_DIR + "/data/processed_continuous",
    "reports": DRIVE_DIR + "/reports",
    "outputs/models": DRIVE_DIR + "/outputs/models",
}
for local, target in links.items():
    os.makedirs(target, exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.isdir(local):
        shutil.rmtree(local)
    os.symlink(target, local)
    print(local, "->", target)

## 7. Build the windows (resumable)
This cuts every recording into 1 s windows every 0.25 s, with the filtering and the quality decision of the
project. Subjects that were already built on Drive are skipped (`--resume`), so after a disconnect this cell is
fast. The first time it can take a while for 40 subjects.

In [ ]:
!python -m scripts.build_windows --filter-mode continuous --session OO --quality-source both --banks fbcca fbcsp --window-sec 1 --stride-sec 0.25 --resume

## 8. Within-subject reference
Each subject is tested with cross-validation inside his or her own session (trial-grouped, no leakage). This is
the number the LOSO result is compared with. Classical methods only (`fbcca`, `lda`); minutes on CPU.

In [ ]:
!python -m scripts.benchmarks.benchmark_window_sweep --data-root data/processed_continuous --windows 1 --stride 0.25 --methods fbcca lda --cv-repeats 5 --out-dir {WITHIN_OUT}

## 9. LOSO, classical methods
`fbcca` (no training), `lda` and `lda_subjnorm` (the latter normalizes each subject with his or her own unlabeled
statistics: a test-time normalization, reported with that label). About 30 to 35 min for 40 subjects, CPU only.
Results go to `LOSO_OUT` on Drive. Rerun the cell after a disconnect: finished folds are skipped.

In [ ]:
!python -m scripts.benchmarks.benchmark_loso --data-root data/processed_continuous --window 1 --stride-sec 0.25 --methods fbcca lda lda_subjnorm --channels common --out-dir {LOSO_OUT}

## 10. LOSO, networks (one network per cell)
Each cell trains one network 40 times (once per held-out subject) on the GPU. Run **one cell at a time**, in any
order, and only the ones you need. Each held-out subject is saved on Drive as soon as it ends: if the session is
cut, run steps 1 to 6 again and then this same cell; it continues where it stopped (`[skip]` lines).
If the output says the device is `cpu`, the GPU is not active (see step 1) and the run would take days: stop it.

Expected time for 40 subjects (RTX 3050-class GPU; a T4 is of similar order): `ccnn` about 1.5 h,
`eegnet` about 4.9 h, `compact_eegnet` about 12.5 h. The last one needs several sessions; that is expected.
A warning `degenerate network folds` means a fold stopped at once or never learned; keep it in mind when reading the results.

In [ ]:
# Network 1: ccnn (about 1.5 h for 40 subjects)
!python -m scripts.benchmarks.benchmark_loso --data-root data/processed_continuous --window 1 --stride-sec 0.25 --methods ccnn --channels common --device auto --save-models --models-dir {MODELS_DIR} --out-dir {LOSO_OUT}

In [ ]:
# Network 2: eegnet (about 4.9 h for 40 subjects)
!python -m scripts.benchmarks.benchmark_loso --data-root data/processed_continuous --window 1 --stride-sec 0.25 --methods eegnet --channels common --device auto --save-models --models-dir {MODELS_DIR} --out-dir {LOSO_OUT}

In [ ]:
# Network 3: compact_eegnet (about 12.5 h for 40 subjects; several sessions)
!python -m scripts.benchmarks.benchmark_loso --data-root data/processed_continuous --window 1 --stride-sec 0.25 --methods compact --channels common --device auto --save-models --models-dir {MODELS_DIR} --out-dir {LOSO_OUT}

## 10b. LOSO, connectivity GNN (P23; CPU is enough)
**What it is.** A small graph neural network. Each window becomes a graph whose nodes are the EEG channels and whose
edges are the connectivity between channels (coherence); the network learns which edges to trust. Two variants:
`nodes+F` (node features plus the FBCCA scores) and `nodes+coh+F` (the same plus the coherence edges). Each fold
is trained with three seeds and the three predictions are averaged. Early stopping uses 20 % of the training
*trials*, never the held-out subject.

**Two normalizations (one per cell).**
- `pooled`: the statistics of the training windows only. Method name `gnn_<variant>`.
- `subject` (method name ends in `_subjnorm`): each subject is z-scored with his or her **own whole-session
  statistics, without labels**. This is a *test-time normalization* that sees the held-out subject's whole session,
  so it is an **offline upper bound**, not an online result.

**Within reference.** The same windows with a trial-grouped 5-fold cross-validation inside each subject (pooled only).
It is the number the LOSO result is compared with. A high training accuracy next to a near-chance test accuracy
is a known pattern (overfitting to the training subjects); read the `folds.csv` diagnosis.

**Important.** This section needs `benchmark_gnn_loso` (P23), which `main` contains: keep `BRANCH = "main"` in step 3
(if you changed it, set it back and rerun steps 4 to 7) before using it.

**Expected times for the 5 subjects S01 to S05** (2 CPU threads; the GPU is not faster for a network this small):
smoke test about 1 min; within reference (50 folds) about 7 to 9 min; each LOSO cell (5 folds) about 3 min,
the four cells about 12 to 13 min. For 40 subjects multiply by about 8.

**Resuming.** Each fold is saved on Drive as soon as it ends; rerun the same cell after a disconnect and finished folds
are skipped (`[skip]`). The two output folders below are new and separate from the P21 results. `--threads` and
`--device` are stored in each fold: keep them as they are in every cell of an output folder, and use a new
folder name if you change any setting (a folder refuses folds computed with another configuration).

In [ ]:
GNN_LOSO_OUT = "reports/loso_gnn_w1_colab"      # own folder: never reuse a P21 folder
GNN_WITHIN_OUT = "reports/within_gnn_w1_colab"
assert GNN_LOSO_OUT != LOSO_OUT and GNN_WITHIN_OUT != WITHIN_OUT
print(GNN_LOSO_OUT, GNN_WITHIN_OUT)

### 10b.1 Smoke test (about 1 min)
Synthetic subjects only, no recordings. It must end with `PASS`; if not, do not run the cells below.

In [ ]:
!python -m scripts.connectivity.graph_modeling.benchmark_gnn_loso --smoke-test --threads 2 --device cpu

### 10b.2 Within-subject reference (about 7 to 9 min for 5 subjects)
Both variants, pooled, 3 seeds, 5 folds inside each subject.

In [ ]:
!python -m scripts.connectivity.graph_modeling.benchmark_gnn_loso --data-root data/processed_continuous --window 1 --stride-sec 0.25 --channels common --seeds 3 --threads 2 --device cpu --protocol within --variants nodes+F nodes+coh+F --norm pooled --out-dir {GNN_WITHIN_OUT}

### 10b.3 LOSO (four cells, about 3 min each for 5 subjects)
One variant and one normalization per cell, in any order. All four write to `GNN_LOSO_OUT` and share the same
settings (`--threads 2 --device cpu`). The `subject` cells are the offline upper bound explained above.

In [ ]:
# gnn_nodes+F (about 3 min for 5 subjects)
!python -m scripts.connectivity.graph_modeling.benchmark_gnn_loso --data-root data/processed_continuous --window 1 --stride-sec 0.25 --channels common --seeds 3 --threads 2 --device cpu --protocol loso --variants nodes+F --norm pooled --out-dir {GNN_LOSO_OUT}

In [ ]:
# gnn_nodes+F_subjnorm (about 3 min for 5 subjects)
!python -m scripts.connectivity.graph_modeling.benchmark_gnn_loso --data-root data/processed_continuous --window 1 --stride-sec 0.25 --channels common --seeds 3 --threads 2 --device cpu --protocol loso --variants nodes+F --norm subject --out-dir {GNN_LOSO_OUT}

In [ ]:
# gnn_nodes+coh+F (about 3 min for 5 subjects)
!python -m scripts.connectivity.graph_modeling.benchmark_gnn_loso --data-root data/processed_continuous --window 1 --stride-sec 0.25 --channels common --seeds 3 --threads 2 --device cpu --protocol loso --variants nodes+coh+F --norm pooled --out-dir {GNN_LOSO_OUT}

In [ ]:
# gnn_nodes+coh+F_subjnorm (about 3 min for 5 subjects)
!python -m scripts.connectivity.graph_modeling.benchmark_gnn_loso --data-root data/processed_continuous --window 1 --stride-sec 0.25 --channels common --seeds 3 --threads 2 --device cpu --protocol loso --variants nodes+coh+F --norm subject --out-dir {GNN_LOSO_OUT}

### 10b.4 Read the results
Open `summary.csv` (accuracy per subject and method, with the `Protocol` and `Normalization` columns) and `folds.csv`
(train, validation and test accuracy, and the diagnosis) in `GNN_LOSO_OUT` and `GNN_WITHIN_OUT` on Drive. The
GNN rows are interpreted next to the `benchmark_loso` rows (`fbcca`, `lda`, `lda_subjnorm`) in the phase report.

## 11. Within-subject versus LOSO, per subject and mean
The table lists the accuracy (%) of each method within the subject's own session and when the subject is held out.
Read it as a direction, not as a target: LOSO is expected to be at or below the within value for trained
methods, and `fbcca` (nothing is trained) should coincide when both use the same channels. The within reference
uses the channels each subject's own quality check kept, LOSO uses the channels kept in every subject
(`common`), so a subject with a different set can differ slightly. The table also lists `lda_subjnorm`
(unsupervised, test-time normalization) and the networks that were run.

In [ ]:
import pandas as pd
within = pd.read_csv(WITHIN_OUT + "/summary.csv")
loso = pd.read_csv(LOSO_OUT + "/summary.csv")
w = within.pivot_table(index="Subject", columns="Method", values="Acc (%)", aggfunc="first").add_prefix("within ")
l = loso.pivot_table(index="Subject", columns="Method", values="Acc (%)", aggfunc="first").add_prefix("LOSO ")
table = w.join(l, how="outer")
subj = table[~table.index.isin(["Mean", "SD"])]
subj = subj.loc[sorted(subj.index, key=lambda s: int(s[1:]))]
mean = subj.mean(numeric_only=True).rename("Mean (subjects)")
pd.set_option("display.width", 200)
print(pd.concat([subj, mean.to_frame().T]).round(2).to_string())
print()
print("within source:", within["Data source"].iloc[0])
print("LOSO source:  ", loso["Data source"].iloc[0])

## 12. Release the runtime
Always finish with this cell: an idle session keeps using your credits until it times out. Your results are
already on Drive (`DRIVE_DIR/reports/`). Download `summary.csv`, `per_subject.csv` and `folds.csv` of `LOSO_OUT`
from Drive before you run it if you want a local copy.

In [ ]:
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print("Could not release automatically (", e, "). Use Runtime > Disconnect and delete runtime.")